# چت محلی

اگر خطای قبلی دارید: **Runtime → Disconnect and delete runtime**، بعد GPU را روشن کنید و Run all بزنید.

نصب CUDA ممکن است چند دقیقه طول بکشد.


In [ ]:
import os, sys, subprocess, shutil

def run(cmd, env=None):
    print('>>', cmd if isinstance(cmd, str) else ' '.join(cmd))
    subprocess.check_call(cmd if isinstance(cmd, list) else cmd, shell=isinstance(cmd, str), env=env)

subprocess.call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'llama-cpp-python'])
for p in list(sys.path):
    d = os.path.join(p, 'llama_cpp')
    if os.path.isdir(d):
        shutil.rmtree(d, ignore_errors=True)

run([sys.executable, '-m', 'pip', 'install', '-q', 'gradio', 'ddgs', 'huggingface_hub'])

for cand in ['/usr/local/cuda/lib64', '/usr/local/cuda/lib', '/usr/lib64-nvidia']:
    if os.path.isdir(cand):
        os.environ['LD_LIBRARY_PATH'] = cand + ':' + os.environ.get('LD_LIBRARY_PATH', '')
        print('LD +=', cand)

ok = False
try:
    env = os.environ.copy()
    env['CMAKE_ARGS'] = '-DGGML_CUDA=on'
    env['FORCE_CMAKE'] = '1'
    print('Building llama-cpp-python with CUDA (several minutes)...')
    run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-cache-dir', 'llama-cpp-python'], env=env)
    if 'llama_cpp' in sys.modules:
        del sys.modules['llama_cpp']
    import llama_cpp
    print('OK CUDA build', llama_cpp.__version__)
    ok = True
except Exception as e:
    print('CUDA build failed:', e)
    subprocess.call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'llama-cpp-python'])

if not ok:
    for tag in ('cu122', 'cu121', 'cu118', 'cu124'):
        try:
            run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-cache-dir', 'llama-cpp-python', '--extra-index-url', f'https://abetlen.github.io/llama-cpp-python/whl/{tag}'])
            if 'llama_cpp' in sys.modules:
                del sys.modules['llama_cpp']
            import llama_cpp
            print('OK wheel', tag, llama_cpp.__version__)
            ok = True
            break
        except Exception as e:
            print('wheel fail', tag, e)
            subprocess.call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'llama-cpp-python'])

if not ok:
    env = os.environ.copy()
    env['CMAKE_ARGS'] = '-DGGML_CUDA=off'
    env['FORCE_CMAKE'] = '1'
    print('CPU fallback...')
    run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-cache-dir', 'llama-cpp-python'], env=env)
    import llama_cpp
    print('OK CPU', llama_cpp.__version__)

import llama_cpp
print('ready', llama_cpp.__version__)


In [ ]:
import os
import torch
from huggingface_hub import hf_hub_download

print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
if not torch.cuda.is_available():
    print('هشدار: GPU روشن نیست')

path = hf_hub_download(
    repo_id='mradermacher/Huihui-Qwen3-8B-abliterated-v2-GGUF',
    filename='Huihui-Qwen3-8B-abliterated-v2.Q4_K_M.gguf',
)
print('model', path, round(os.path.getsize(path) / 1e9, 2), 'GB')


In [ ]:
import re
from threading import Lock
import gradio as gr
from llama_cpp import Llama
import torch

EFFORTS = {
    'کم': {'max_tokens': 384, 'temperature': 0.35, 'think': False},
    'معمولی': {'max_tokens': 800, 'temperature': 0.6, 'think': False},
    'زیاد': {'max_tokens': 1500, 'temperature': 0.7, 'think': True},
    'حداکثر': {'max_tokens': 2600, 'temperature': 0.85, 'think': True},
}

DEFAULT_SYSTEM = (
    'تو یک دستیار محلی و مستقیم هستی. به زبان کاربر جواب بده، کوتاه و مشخص، بدون موعظه. '
    'محتوای جنسی مربوط به افراد زیر ۱۸ سال نساز. '
    'برای حمله خشونت‌آمیز، مواد منفجره یا کلاهبرداری راهنمای اجرایی نده. '
    'برای بقیه موضوع‌ها واضح جواب بده. '
    'اگر نتیجه جستجو داده شد، فقط بخش مرتبط را استفاده کن و منبع را بنویس.'
)

_llm = None
_lock = Lock()

def load_model(model_path):
    global _llm
    layers = -1 if torch.cuda.is_available() else 0
    _llm = Llama(model_path=model_path, n_gpu_layers=layers, n_ctx=8192, n_batch=512, verbose=False)
    return 'مدل آماده است.'

def _search(query, n):
    from ddgs import DDGS
    hits = DDGS().text(query[:240], max_results=int(n)) or []
    lines = []
    for i, hit in enumerate(hits, 1):
        title = hit.get('title') or ''
        body = hit.get('body') or ''
        url = hit.get('href') or hit.get('url') or ''
        lines.append(f'[{i}] {title}\n{body}\n{url}')
    return '\n\n'.join(lines) if lines else 'نتیجه‌ای پیدا نشد.'

def _strip_think(text, show):
    if show:
        return text
    return re.sub(r'<think>.*?</think>', '', text, flags=re.S).strip()

def reply(message, history, effort, deep, n_results, system, show_think):
    text = (message or '').strip()
    if not text:
        return history or [], ''
    if _llm is None:
        raise gr.Error('مدل هنوز بارگذاری نشده.')
    cfg = EFFORTS.get(effort, EFFORTS['معمولی'])
    user = text
    if deep:
        try:
            found = _search(text, n_results)
        except Exception as exc:
            found = f'جستجو ناموفق بود: {exc}'
        user = f'نتایج DeepSearch:\n{found}\n\nسؤال کاربر:\n{text}'
    user += '\n/think' if cfg['think'] else '\n/no_think'
    messages = [{'role': 'system', 'content': (system or DEFAULT_SYSTEM).strip()}]
    for turn in history or []:
        if isinstance(turn, dict) and turn.get('role') in ('user', 'assistant'):
            messages.append({'role': turn['role'], 'content': str(turn.get('content') or '')})
    messages.append({'role': 'user', 'content': user})
    with _lock:
        out = _llm.create_chat_completion(messages=messages, max_tokens=cfg['max_tokens'], temperature=cfg['temperature'], top_p=0.9, repeat_penalty=1.05)
    answer = _strip_think(out['choices'][0]['message']['content'] or '', show_think)
    updated = list(history or []) + [{'role': 'user', 'content': text}, {'role': 'assistant', 'content': answer or 'پاسخی تولید نشد.'}]
    return updated, ''

status = load_model(path)
with gr.Blocks(title='چت محلی') as demo:
    gr.Markdown('## چت محلی\n**ایفورت** و **DeepSearch** را تنظیم کنید.')
    gr.Markdown(status)
    try:
        chat = gr.Chatbot(height=480, type='messages', label='گفتگو')
    except TypeError:
        chat = gr.Chatbot(height=480, label='گفتگو')
    message = gr.Textbox(label='پیام', lines=3, autofocus=True)
    with gr.Row():
        effort = gr.Radio(list(EFFORTS), value='معمولی', label='ایفورت')
        deep = gr.Checkbox(value=False, label='DeepSearch')
        n_results = gr.Slider(3, 8, value=5, step=1, label='تعداد نتیجه')
        show_think = gr.Checkbox(value=False, label='نمایش فکر')
    system = gr.Textbox(label='دستور سیستم', value=DEFAULT_SYSTEM, lines=3)
    with gr.Row():
        send = gr.Button('ارسال', variant='primary')
        clear = gr.Button('پاک کردن')
    send.click(reply, [message, chat, effort, deep, n_results, system, show_think], [chat, message])
    message.submit(reply, [message, chat, effort, deep, n_results, system, show_think], [chat, message])
    clear.click(lambda: ([], ''), outputs=[chat, message])
demo.launch(share=True, server_name='0.0.0.0', server_port=7860, prevent_thread_lock=False, show_error=True)
